In [1]:
from __future__ import annotations

In [2]:
from dataclasses import dataclass
import csv
import json
import math
import os
import warnings
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import DatasetDict, get_dataset_config_names, load_dataset
from huggingface_hub import snapshot_download
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, T5EncoderModel, T5ForConditionalGeneration
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    coverage_error,
    hamming_loss,
    jaccard_score,
    label_ranking_average_precision_score,
    label_ranking_loss,
    precision_recall_fscore_support,
    roc_auc_score,
    zero_one_loss,
)
from tqdm.auto import tqdm
from transformers.modeling_outputs import BaseModelOutput

In [3]:
@dataclass
class T5EncoderOutput:
    last_hidden_state: torch.Tensor


@dataclass
class T5DecoderOutput:
    last_hidden_state: torch.Tensor
    all_hidden_states: Optional[Tuple[torch.Tensor, ...]] = None


@dataclass
class FactorVAEOutput:
    z: torch.Tensor
    mu: torch.Tensor
    logvar: torch.Tensor
    scalar_z: torch.Tensor
    vector_z: torch.Tensor
    scalar_mu: torch.Tensor
    vector_mu: torch.Tensor
    scalar_logvar: torch.Tensor
    vector_logvar: torch.Tensor


@dataclass
class T5FactorVAEModelOutput:
    t5_encoder_sequence: torch.Tensor
    attention_weights: torch.Tensor
    pooled_scalar_tensor: torch.Tensor
    vae: FactorVAEOutput
    vae_decoded_sequence: torch.Tensor
    decoder_memory: torch.Tensor
    t5_decoder_sequence: Optional[torch.Tensor] = None
    classification_logits: Optional[torch.Tensor] = None
    loss: Optional[torch.Tensor] = None
    loss_terms: Optional[Dict[str, torch.Tensor]] = None


In [4]:
class ScalarLatentAttentionPooling(nn.Module):
    def __init__(
        self,
        num_scalar_factors: int,
        source_dim: int,
        num_heads: int = 4,
        pooling_mode: str = "per_scalar_dim",
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        if pooling_mode not in {"per_scalar_dim", "joint_scalar_vector"}:
            raise ValueError(
                f"Unsupported pooling_mode={pooling_mode}. Expected one of: per_scalar_dim, joint_scalar_vector."
            )
        self.num_scalar_factors = num_scalar_factors
        self.source_dim = source_dim
        self.num_heads = num_heads
        self.pooling_mode = pooling_mode
        self.dropout = nn.Dropout(dropout)

        if pooling_mode == "per_scalar_dim":
            self.score_weight = nn.Parameter(
                torch.empty(num_scalar_factors, num_heads, source_dim)
            )
            self.score_bias = nn.Parameter(torch.zeros(num_scalar_factors, num_heads))
        else:
            self.score_weight = nn.Parameter(torch.empty(num_heads, source_dim))
            self.score_bias = nn.Parameter(torch.zeros(num_heads))
        nn.init.xavier_uniform_(self.score_weight)

    def forward(
        self,
        source_sequence: torch.Tensor,
        scalar_sequence: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        source_sequence = self.dropout(source_sequence)

        if self.pooling_mode == "per_scalar_dim":
            logits = torch.einsum(
                "bsd,nhd->bnhs",
                source_sequence,
                self.score_weight,
            ) + self.score_bias.unsqueeze(0).unsqueeze(-1)
        else:
            shared_logits = torch.einsum(
                "bsd,hd->bhs",
                source_sequence,
                self.score_weight,
            ) + self.score_bias.unsqueeze(0).unsqueeze(-1)
            logits = shared_logits.unsqueeze(1).expand(
                -1,
                self.num_scalar_factors,
                -1,
                -1,
            )

        if attention_mask is not None:
            keep_mask = attention_mask.bool()
            logits = logits.masked_fill(
                ~keep_mask.unsqueeze(1).unsqueeze(1),
                torch.finfo(source_sequence.dtype).min,
            )

        weights = torch.softmax(logits, dim=-1)
        values = scalar_sequence.transpose(1, 2).unsqueeze(2)
        pooled = torch.sum(weights * values, dim=-1)
        return pooled, weights

In [5]:
class T5EncoderBackbone(nn.Module):
    def __init__(
        self,
        model_name: str = "google/flan-t5-base",
        use_lora: bool = False,
        lora_r: int = 16,
        lora_alpha: int = 32,
        lora_dropout: float = 0.0,
        lora_target_modules: Sequence[str] = ("q", "v"),
    ) -> None:
        super().__init__()
        model = T5EncoderModel.from_pretrained(model_name)
        if use_lora:
            model = get_peft_model(
                model,
                LoraConfig(
                    task_type=TaskType.FEATURE_EXTRACTION,
                    r=lora_r,
                    lora_alpha=lora_alpha,
                    lora_dropout=lora_dropout,
                    target_modules=list(lora_target_modules),
                    bias="none",
                ),
            )
        self.model = model
        self.hidden_size = model.config.d_model

    def forward(
        self,
        input_ids: torch.LongTensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> T5EncoderOutput:
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        return T5EncoderOutput(last_hidden_state=outputs.last_hidden_state)
    

class T5DecoderBackbone(nn.Module):
    def __init__(
        self,
        model_name: str = "google/flan-t5-base",
        use_lora: bool = False,
        lora_r: int = 16,
        lora_alpha: int = 32,
        lora_dropout: float = 0.0,
        lora_target_modules: Sequence[str] = ("q", "v"),
    ) -> None:
        super().__init__()
        model = T5ForConditionalGeneration.from_pretrained(model_name)
        if use_lora:
            model = get_peft_model(
                model,
                LoraConfig(
                    task_type=TaskType.SEQ_2_SEQ_LM,
                    r=lora_r,
                    lora_alpha=lora_alpha,
                    lora_dropout=lora_dropout,
                    target_modules=list(lora_target_modules),
                    bias="none",
                ),
            )
        self.model = model
        self.hidden_size = model.config.d_model

    def _decoder_start_tokens(
        self,
        batch_size: int,
        device: torch.device,
    ) -> torch.LongTensor:
        start_token_id = self.model.config.decoder_start_token_id
        return torch.full(
            (batch_size, 1),
            fill_value=start_token_id,
            dtype=torch.long,
            device=device,
        )

    def forward(
        self,
        encoder_hidden_states: torch.Tensor,
        encoder_attention_mask: Optional[torch.Tensor] = None,
        decoder_input_ids: Optional[torch.LongTensor] = None,
        decoder_attention_mask: Optional[torch.Tensor] = None,
        decoder_inputs_embeds: Optional[torch.Tensor] = None,
    ) -> T5DecoderOutput:
        batch_size = encoder_hidden_states.size(0)
        if decoder_input_ids is None and decoder_inputs_embeds is None:
            decoder_input_ids = self._decoder_start_tokens(
                batch_size=batch_size,
                device=encoder_hidden_states.device,
            )

        outputs = self.model(
            attention_mask=encoder_attention_mask,
            encoder_outputs=BaseModelOutput(last_hidden_state=encoder_hidden_states),
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            decoder_inputs_embeds=decoder_inputs_embeds,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )
        decoder_states = outputs.decoder_hidden_states
        if decoder_states is None:
            raise RuntimeError("Expected decoder hidden states, but the model returned None.")
        return T5DecoderOutput(
            last_hidden_state=decoder_states[-1],
            all_hidden_states=tuple(decoder_states),
        )

In [6]:
class FactorVAEEncoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_scalar_factors: int,
        vector_latent_dim: int,
        hidden_dim: int = 1024,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.num_scalar_factors = num_scalar_factors
        self.vector_latent_dim = vector_latent_dim
        self.total_latent_dim = num_scalar_factors + vector_latent_dim

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.mu = nn.Linear(hidden_dim, self.total_latent_dim)
        self.logvar = nn.Linear(hidden_dim, self.total_latent_dim)

    def split(self, tensor: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        scalar = tensor[..., : self.num_scalar_factors]
        vector = tensor[..., self.num_scalar_factors :]
        return scalar, vector

    def reparameterize(
        self,
        mu: torch.Tensor,
        logvar: torch.Tensor,
        sample: bool = True,
    ) -> torch.Tensor:
        if not sample:
            return mu
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x: torch.Tensor, sample: bool = True) -> FactorVAEOutput:
        hidden = self.net(x)
        mu = self.mu(hidden)
        logvar = self.logvar(hidden)
        z = self.reparameterize(mu=mu, logvar=logvar, sample=sample)

        scalar_z, vector_z = self.split(z)
        scalar_mu, vector_mu = self.split(mu)
        scalar_logvar, vector_logvar = self.split(logvar)

        return FactorVAEOutput(
            z=z,
            mu=mu,
            logvar=logvar,
            scalar_z=scalar_z,
            vector_z=vector_z,
            scalar_mu=scalar_mu,
            vector_mu=vector_mu,
            scalar_logvar=scalar_logvar,
            vector_logvar=vector_logvar,
        )


class FactorVAEDecoder(nn.Module):
    def __init__(
        self,
        output_dim: int,
        num_scalar_factors: int,
        vector_latent_dim: int,
        hidden_dim: int = 1024,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        total_latent_dim = num_scalar_factors + vector_latent_dim
        self.net = nn.Sequential(
            nn.Linear(total_latent_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(
        self,
        scalar_latent: Optional[torch.Tensor] = None,
        vector_latent: Optional[torch.Tensor] = None,
        z: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if z is None:
            if scalar_latent is None or vector_latent is None:
                raise ValueError(
                    "Provide either z or both scalar_latent and vector_latent to the FactorVAEDecoder."
                )
            z = torch.cat([scalar_latent, vector_latent], dim=-1)
        return self.net(z)

In [7]:
class T5FactorVAEModel(nn.Module):
    def __init__(
        self,
        model_name: str = "google/flan-t5-base",
        num_scalar_factors: int = 8,
        vector_latent_dim: int = 64,
        vae_hidden_dim: int = 1024,
        use_lora: bool = False,
        lora_r: int = 16,
        lora_alpha: int = 32,
        lora_dropout: float = 0.0,
        lora_target_modules: Sequence[str] = ("q", "v"),
        pooling_dropout: float = 0.1,
        latent_pool_heads: int = 4,
        attention_source: str = "encoder_sequence",
        pooling_mode: str = "per_scalar_dim",
        use_skip_connection: bool = True,
        vae_dropout: float = 0.1,
        num_labels: int = 28,
        classifier_dropout: float = 0.1,
        classifier_mode: str = "per_emotion_mlp",
        per_emotion_hidden_dim: int = 32,
        classifier_uses_mu: bool = True,
        run_frozen_t5_decoder: bool = False,
    ) -> None:
        super().__init__()
        self.num_scalar_factors = num_scalar_factors
        self.vector_latent_dim = vector_latent_dim
        self.attention_source = attention_source
        self.pooling_mode = pooling_mode
        self.use_skip_connection = use_skip_connection
        self.num_labels = num_labels
        self.classifier_mode = classifier_mode
        self.classifier_uses_mu = classifier_uses_mu
        self.run_frozen_t5_decoder = run_frozen_t5_decoder

        valid_sources = {"scalar_only", "vector_only", "latent_full", "encoder_sequence"}
        if attention_source not in valid_sources:
            raise ValueError(
                f"Unsupported attention_source={attention_source}. Expected one of: scalar_only, vector_only, latent_full, encoder_sequence."
            )
        valid_classifier_modes = {"joint_mlp", "per_emotion_mlp"}
        if classifier_mode not in valid_classifier_modes:
            raise ValueError(
                f"Unsupported classifier_mode={classifier_mode}. Expected one of: joint_mlp, per_emotion_mlp."
            )
        if classifier_mode == "per_emotion_mlp" and num_scalar_factors != num_labels:
            raise ValueError(
                "For per_emotion_mlp, num_scalar_factors must equal num_labels so each emotion has its own pooled slot."
            )

        self.t5_encoder = T5EncoderBackbone(
            model_name=model_name,
            use_lora=use_lora,
            lora_r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            lora_target_modules=lora_target_modules,
        )
        self.t5_decoder = T5DecoderBackbone(
            model_name=model_name,
            use_lora=use_lora,
            lora_r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            lora_target_modules=lora_target_modules,
        )

        self.hidden_size = self.t5_encoder.hidden_size

        self.vae_encoder = FactorVAEEncoder(
            input_dim=self.hidden_size,
            num_scalar_factors=num_scalar_factors,
            vector_latent_dim=vector_latent_dim,
            hidden_dim=vae_hidden_dim,
            dropout=vae_dropout,
        )

        source_dim_map = {
            "scalar_only": num_scalar_factors,
            "vector_only": vector_latent_dim,
            "latent_full": num_scalar_factors + vector_latent_dim,
            "encoder_sequence": self.hidden_size,
        }
        self.scalar_attention_pool = ScalarLatentAttentionPooling(
            num_scalar_factors=num_scalar_factors,
            source_dim=source_dim_map[attention_source],
            num_heads=latent_pool_heads,
            pooling_mode=pooling_mode,
            dropout=pooling_dropout,
        )

        self.vae_decoder = FactorVAEDecoder(
            output_dim=self.hidden_size,
            num_scalar_factors=num_scalar_factors,
            vector_latent_dim=vector_latent_dim,
            hidden_dim=vae_hidden_dim,
            dropout=vae_dropout,
        )
        self.memory_norm = nn.LayerNorm(self.hidden_size)
        self.memory_gate = nn.Linear(self.hidden_size, self.hidden_size)

        classifier_input_dim = num_scalar_factors * latent_pool_heads
        if classifier_mode == "joint_mlp":
            self.classifier = nn.Sequential(
                nn.Dropout(classifier_dropout),
                nn.Linear(classifier_input_dim, num_labels),
            )
        else:
            self.emotion_classifiers = nn.ModuleList(
                [
                    nn.Sequential(
                        nn.Dropout(classifier_dropout),
                        nn.Linear(latent_pool_heads, per_emotion_hidden_dim),
                        nn.GELU(),
                        nn.Linear(per_emotion_hidden_dim, 1),
                    )
                    for _ in range(num_labels)
                ]
            )
        self.classification_loss_fn = nn.BCEWithLogitsLoss()

    def set_classification_pos_weight(self, pos_weight: torch.Tensor) -> None:
        self.classification_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def _get_attention_source_sequence(
        self,
        vae_out: FactorVAEOutput,
        t5_encoder_sequence: torch.Tensor,
    ) -> torch.Tensor:
        if self.classifier_uses_mu:
            if self.attention_source == "scalar_only":
                return vae_out.scalar_mu
            if self.attention_source == "vector_only":
                return vae_out.vector_mu
            if self.attention_source == "latent_full":
                return vae_out.mu
            return t5_encoder_sequence

        if self.attention_source == "scalar_only":
            return vae_out.scalar_z
        if self.attention_source == "vector_only":
            return vae_out.vector_z
        if self.attention_source == "latent_full":
            return vae_out.z
        return t5_encoder_sequence

    def fuse_memory(
        self,
        t5_encoder_sequence: torch.Tensor,
        decoded_full: torch.Tensor,
    ) -> torch.Tensor:
        gate = torch.sigmoid(self.memory_gate(decoded_full))
        fused = t5_encoder_sequence + gate * decoded_full
        return self.memory_norm(fused)

    def encode(
        self,
        input_ids: torch.LongTensor,
        attention_mask: Optional[torch.Tensor] = None,
        sample_posterior: bool = True,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, FactorVAEOutput]:
        t5_encoder_sequence = self.t5_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state
        vae_out = self.vae_encoder(t5_encoder_sequence, sample=sample_posterior)

        source_sequence = self._get_attention_source_sequence(
            vae_out=vae_out,
            t5_encoder_sequence=t5_encoder_sequence,
        )
        scalar_sequence = vae_out.scalar_mu if self.classifier_uses_mu else vae_out.scalar_z
        pooled_scalar_tensor, attn_weights = self.scalar_attention_pool(
            source_sequence=source_sequence,
            scalar_sequence=scalar_sequence,
            attention_mask=attention_mask,
        )
        return t5_encoder_sequence, attn_weights, pooled_scalar_tensor, vae_out

    def decode(
        self,
        vae_out: FactorVAEOutput,
        t5_encoder_sequence: torch.Tensor,
        encoder_attention_mask: Optional[torch.Tensor] = None,
        decoder_input_ids: Optional[torch.LongTensor] = None,
        decoder_attention_mask: Optional[torch.Tensor] = None,
        decoder_inputs_embeds: Optional[torch.Tensor] = None,
    ) -> Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor]:
        vae_decoded_sequence = self.vae_decoder(z=vae_out.z)
        if self.use_skip_connection:
            decoder_memory = self.fuse_memory(t5_encoder_sequence, vae_decoded_sequence)
        else:
            decoder_memory = vae_decoded_sequence

        if not self.run_frozen_t5_decoder:
            return None, vae_decoded_sequence, decoder_memory

        decoded = self.t5_decoder(
            encoder_hidden_states=decoder_memory,
            encoder_attention_mask=encoder_attention_mask,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            decoder_inputs_embeds=decoder_inputs_embeds,
        )
        return decoded.last_hidden_state, vae_decoded_sequence, decoder_memory

    def _classification_logits(
        self,
        pooled_scalar_tensor: torch.Tensor,
        t5_encoder_sequence: torch.Tensor,
        attention_mask: Optional[torch.Tensor],
    ) -> torch.Tensor:
        if self.classifier_mode == "joint_mlp":
            features = pooled_scalar_tensor.flatten(start_dim=1)
            return self.classifier(features)
        logits = [
            head(pooled_scalar_tensor[:, idx, :])
            for idx, head in enumerate(self.emotion_classifiers)
        ]
        return torch.cat(logits, dim=-1)

    def _compute_losses(
        self,
        classification_logits: torch.Tensor,
        t5_encoder_sequence: torch.Tensor,
        vae_decoded_sequence: torch.Tensor,
        vae_out: FactorVAEOutput,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        classification_weight: float = 1.0,
        kl_weight: float = 0.01,
        recon_weight: float = 0.1,
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        if attention_mask is None:
            recon_loss = F.mse_loss(vae_decoded_sequence, t5_encoder_sequence)
        else:
            mask = attention_mask.to(t5_encoder_sequence.dtype).unsqueeze(-1)
            sq_error = (vae_decoded_sequence - t5_encoder_sequence).pow(2)
            denom = mask.sum().clamp_min(1.0) * sq_error.size(-1)
            recon_loss = (sq_error * mask).sum() / denom
        kl_per_token = -0.5 * (
            1 + vae_out.logvar - vae_out.mu.pow(2) - vae_out.logvar.exp()
        ).sum(dim=-1)

        if attention_mask is None:
            kl_loss = kl_per_token.mean()
        else:
            mask = attention_mask.to(kl_per_token.dtype)
            kl_loss = (kl_per_token * mask).sum() / mask.sum().clamp_min(1.0)

        total_loss = (recon_weight * recon_loss) + (kl_weight * kl_loss)
        loss_terms: Dict[str, torch.Tensor] = {
            "reconstruction": recon_loss.detach(),
            "kl": kl_loss.detach(),
        }

        if labels is not None:
            classification_loss = self.classification_loss_fn(
                classification_logits,
                labels.float(),
            )
            total_loss = total_loss + (classification_weight * classification_loss)
            loss_terms["classification"] = classification_loss.detach()

        return total_loss, loss_terms

    def forward(
        self,
        input_ids: torch.LongTensor,
        attention_mask: Optional[torch.Tensor] = None,
        decoder_input_ids: Optional[torch.LongTensor] = None,
        decoder_attention_mask: Optional[torch.Tensor] = None,
        decoder_inputs_embeds: Optional[torch.Tensor] = None,
        sample_posterior: bool = True,
        labels: Optional[torch.Tensor] = None,
        classification_weight: float = 1.0,
        kl_weight: float = 0.01,
        recon_weight: float = 0.1,
    ) -> T5FactorVAEModelOutput:
        (
            t5_encoder_sequence,
            attn_weights,
            pooled_scalar_tensor,
            vae_out,
        ) = self.encode(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sample_posterior=sample_posterior,
        )

        t5_decoder_sequence, vae_decoded_sequence, decoder_memory = self.decode(
            vae_out=vae_out,
            t5_encoder_sequence=t5_encoder_sequence,
            encoder_attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            decoder_inputs_embeds=decoder_inputs_embeds,
        )

        classification_logits = self._classification_logits(
            pooled_scalar_tensor,
            t5_encoder_sequence,
            attention_mask,
        )

        loss = None
        loss_terms = None
        if labels is not None:
            loss, loss_terms = self._compute_losses(
                classification_logits=classification_logits,
                t5_encoder_sequence=t5_encoder_sequence,
                vae_decoded_sequence=vae_decoded_sequence,
                vae_out=vae_out,
                attention_mask=attention_mask,
                labels=labels,
                classification_weight=classification_weight,
                kl_weight=kl_weight,
                recon_weight=recon_weight,
            )

        return T5FactorVAEModelOutput(
            t5_encoder_sequence=t5_encoder_sequence,
            attention_weights=attn_weights,
            pooled_scalar_tensor=pooled_scalar_tensor,
            vae=vae_out,
            vae_decoded_sequence=vae_decoded_sequence,
            decoder_memory=decoder_memory,
            t5_decoder_sequence=t5_decoder_sequence,
            classification_logits=classification_logits,
            loss=loss,
            loss_terms=loss_terms,
        )

In [8]:
GO_EMOTIONS_REPO = "google-research-datasets/go_emotions"


def parse_label_ids(label_value) -> list[int]:
    if label_value is None:
        return []
    if isinstance(label_value, (list, tuple, set)):
        return [int(x) for x in label_value]
    if isinstance(label_value, str):
        text = label_value.strip()
        if text == "":
            return []
        if text.startswith("[") and text.endswith("]"):
            text = text[1:-1]
        return [int(x.strip()) for x in text.split(",") if x.strip() != ""]
    return [int(label_value)]


class GoEmotionsTorchDataset(Dataset):
    def __init__(
        self,
        hf_split,
        tokenizer: AutoTokenizer,
        num_labels: int,
        max_length: int = 48,
    ) -> None:
        self.num_labels = num_labels
        texts = [str(x) for x in hf_split["text"]]
        enc = tokenizer(
            texts,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]

        labels = []
        for raw_labels in hf_split["labels"]:
            label_ids = parse_label_ids(raw_labels)
            target = torch.zeros(num_labels, dtype=torch.float32)
            for label_id in label_ids:
                if 0 <= label_id < num_labels:
                    target[label_id] = 1.0
            labels.append(target)
        self.labels = torch.stack(labels, dim=0)

    def __len__(self) -> int:
        return self.input_ids.size(0)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }


def compute_pos_weight(label_matrix: torch.Tensor, max_ratio: float = 20.0) -> torch.Tensor:
    pos = label_matrix.sum(dim=0)
    neg = label_matrix.size(0) - pos
    pos_weight = neg / pos.clamp_min(1.0)
    return pos_weight.clamp(max=max_ratio)


repo_dir = Path(snapshot_download(repo_id=GO_EMOTIONS_REPO, repo_type="dataset"))
available_configs = get_dataset_config_names(GO_EMOTIONS_REPO)
if "simplified" not in available_configs:
    raise RuntimeError(f"Expected simplified config in {available_configs}")
dataset_config = "simplified"

go_emotions: DatasetDict = load_dataset(GO_EMOTIONS_REPO, dataset_config)
split_names = list(go_emotions.keys())
train_split = go_emotions["train"]
val_split = go_emotions["validation"] if "validation" in go_emotions else go_emotions["test"]
test_split = go_emotions["test"] if "test" in go_emotions else val_split

labels_feature = train_split.features["labels"]
emotion_names = list(labels_feature.feature.names)
num_labels = len(emotion_names)

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
train_dataset = GoEmotionsTorchDataset(train_split, tokenizer, num_labels=num_labels, max_length=48)
val_dataset = GoEmotionsTorchDataset(val_split, tokenizer, num_labels=num_labels, max_length=48)
test_dataset = GoEmotionsTorchDataset(test_split, tokenizer, num_labels=num_labels, max_length=48)
pos_weight = compute_pos_weight(train_dataset.labels)

print(f"downloaded from: {repo_dir}")
print(f"dataset config: {dataset_config}")
print(f"available splits: {split_names}")
print(f"train samples: {len(train_dataset)}")
print(f"validation samples: {len(val_dataset)}")
print(f"test samples: {len(test_dataset)}")
print(f"num_labels: {num_labels}")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

downloaded from: /mnt/data/AI/huggingface/hub/datasets--google-research-datasets--go_emotions/snapshots/add492243ff905527e67aeb8b80c082af02207c3
dataset config: simplified
available splits: ['train', 'validation', 'test']
train samples: 43410
validation samples: 5426
test samples: 5427
num_labels: 28


In [9]:
class FactorVAEDiscriminator(nn.Module):
    def __init__(self, latent_dim: int, hidden_dim: int = 512) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


def permute_latent_dims(z: torch.Tensor) -> torch.Tensor:
    cols = []
    for j in range(z.size(1)):
        idx = torch.randperm(z.size(0), device=z.device)
        cols.append(z[idx, j])
    return torch.stack(cols, dim=1)


def set_requires_grad(module: nn.Module, value: bool) -> None:
    for p in module.parameters():
        p.requires_grad = value


def running_in_notebook() -> bool:
    try:
        from IPython import get_ipython  # type: ignore
        shell = get_ipython()
        if shell is None:
            return False
        return shell.__class__.__name__ == "ZMQInteractiveShell"
    except Exception:
        return False

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
can_use_worker_processes = (
    not running_in_notebook()
    and hasattr(__import__("__main__"), "__file__")
)
num_workers = min(4, os.cpu_count() or 0) if can_use_worker_processes else 0
pin_memory = device.type == "cuda"
loader_kwargs = {
    "num_workers": num_workers,
    "pin_memory": pin_memory,
}
if num_workers > 0:
    loader_kwargs["persistent_workers"] = True

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, **loader_kwargs)

classifier_mode = "per_emotion_mlp"
vae_training_mode = "factorvae"
if vae_training_mode not in {"factorvae", "classic_vae"}:
    raise ValueError("vae_training_mode must be either 'factorvae' or 'classic_vae'.")

num_scalar_factors = num_labels
vector_latent_dim = 512

model = T5FactorVAEModel(
    model_name="google/flan-t5-base",
    num_scalar_factors=num_scalar_factors,
    vector_latent_dim=vector_latent_dim,
    latent_pool_heads=4,
    attention_source="encoder_sequence",
    pooling_mode="per_scalar_dim",
    use_skip_connection=False,
    use_lora=False,
    num_labels=num_labels,
    classifier_mode=classifier_mode,
    classifier_uses_mu=True,
    run_frozen_t5_decoder=False,
).to(device)

if vae_training_mode == "factorvae":
    discriminator = FactorVAEDiscriminator(
        latent_dim=num_scalar_factors + vector_latent_dim,
        hidden_dim=512,
    ).to(device)
    disc_optimizer = torch.optim.Adam(
        discriminator.parameters(),
        lr=5e-5,
        betas=(0.5, 0.9),
    )
    disc_ce = nn.CrossEntropyLoss()
else:
    discriminator = None
    disc_optimizer = None
    disc_ce = None

for param in model.t5_encoder.parameters():
    param.requires_grad = False
for param in model.t5_decoder.parameters():
    param.requires_grad = False

model.t5_encoder.eval()
model.t5_decoder.eval()
model.set_classification_pos_weight(pos_weight.to(device))

optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=2e-4,
)

classification_weight = 2.0
recon_weight = 1.0
kl_weight = 1.0
tc_weight = 6.4 if vae_training_mode == "factorvae" else 0.0
sample_posterior_train = True
sample_posterior_eval = False

print(f"vae_training_mode={vae_training_mode}")
print(f"classifier_mode={classifier_mode}")
print(f"classification_weight={classification_weight}")
print(f"recon_weight={recon_weight}")
print(f"kl_weight={kl_weight}")
print(f"tc_weight={tc_weight}")
print(f"sample_posterior_train={sample_posterior_train}")

Loading weights:   0%|          | 0/111 [00:00<?, ?it/s]

T5EncoderModel LOAD REPORT from: google/flan-t5-base
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


vae_training_mode=factorvae
classifier_mode=per_emotion_mlp
classification_weight=2.0
recon_weight=1.0
kl_weight=1.0
tc_weight=6.4
sample_posterior_train=True


In [11]:
def logits_to_preds(
    logits: torch.Tensor,
    threshold: float | torch.Tensor = 0.5,
) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    if isinstance(threshold, torch.Tensor):
        threshold = threshold.to(device=probs.device, dtype=probs.dtype)
    return (probs >= threshold).to(dtype=torch.float32)



def safe_div(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if abs(denominator) > 1e-12 else 0.0



def safe_metric_call(fn, *args, **kwargs):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            value = fn(*args, **kwargs)
        if isinstance(value, np.ndarray):
            return value.tolist()
        if isinstance(value, np.generic):
            return value.item()
        return value
    except Exception:
        return None



def to_serializable(value: Any) -> Any:
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): to_serializable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_serializable(v) for v in value]
    return value



def compute_macro_f1_from_counts(
    per_label_tp: torch.Tensor,
    per_label_fp: torch.Tensor,
    per_label_fn: torch.Tensor,
) -> float:
    eps = 1e-8
    per_label_precision = per_label_tp / (per_label_tp + per_label_fp + eps)
    per_label_recall = per_label_tp / (per_label_tp + per_label_fn + eps)
    per_label_f1 = 2.0 * per_label_precision * per_label_recall / (
        per_label_precision + per_label_recall + eps
    )
    return float(per_label_f1.mean().item())



def batch_multilabel_stats(
    logits: torch.Tensor,
    labels: torch.Tensor,
    threshold: float | torch.Tensor = 0.5,
) -> Dict[str, torch.Tensor | float]:
    preds = logits_to_preds(logits, threshold=threshold)
    labels = labels.to(dtype=torch.float32)

    per_label_tp = (preds * labels).sum(dim=0).detach().cpu()
    per_label_fp = (preds * (1.0 - labels)).sum(dim=0).detach().cpu()
    per_label_fn = ((1.0 - preds) * labels).sum(dim=0).detach().cpu()

    tp = float(per_label_tp.sum().item())
    fp = float(per_label_fp.sum().item())
    fn = float(per_label_fn.sum().item())
    precision = tp / max(tp + fp, 1e-8)
    recall = tp / max(tp + fn, 1e-8)
    f1 = 2.0 * precision * recall / max(precision + recall, 1e-8)
    macro_f1 = compute_macro_f1_from_counts(
        per_label_tp=per_label_tp,
        per_label_fp=per_label_fp,
        per_label_fn=per_label_fn,
    )
    exact_match = float((preds == labels).all(dim=1).float().mean().item())
    hamming_acc = 1.0 - float((preds != labels).float().mean().item())

    return {
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'macro_f1': macro_f1,
        'exact_match': exact_match,
        'hamming_acc': hamming_acc,
        'per_label_tp': per_label_tp,
        'per_label_fp': per_label_fp,
        'per_label_fn': per_label_fn,
    }



def extract_valid_token_latents(
    z_sequence: torch.Tensor,
    attention_mask: Optional[torch.Tensor],
) -> torch.Tensor:
    if z_sequence.dim() == 2:
        return z_sequence
    flat_z = z_sequence.reshape(-1, z_sequence.size(-1))
    if attention_mask is None:
        return flat_z
    flat_mask = attention_mask.reshape(-1).bool()
    if flat_mask.any():
        return flat_z[flat_mask]
    return flat_z



def decode_from_decoder_memory(
    decoder_memory: torch.Tensor,
    attention_mask: torch.Tensor,
) -> str:
    generated_ids = model.t5_decoder.model.generate(
        encoder_outputs=BaseModelOutput(last_hidden_state=decoder_memory),
        attention_mask=attention_mask,
        max_new_tokens=20,
        num_beams=1,
        do_sample=False,
    )
    return tokenizer.batch_decode(
        generated_ids.detach().cpu(),
        skip_special_tokens=True,
    )[0]



def show_eval_samples_during_training(num_samples: int = 2) -> None:
    model.eval()
    model.t5_encoder.eval()
    model.t5_decoder.eval()
    print('  eval samples:')
    with torch.no_grad():
        for i in range(num_samples):
            row = val_split[i]
            text = str(row['text'])
            gold_ids = parse_label_ids(row['labels'])
            gold_names = [emotion_names[idx] for idx in gold_ids]

            enc = tokenizer(
                text,
                max_length=48,
                padding='max_length',
                truncation=True,
                return_tensors='pt',
            )
            enc = {k: v.to(device) for k, v in enc.items()}

            out = model(
                input_ids=enc['input_ids'],
                attention_mask=enc['attention_mask'],
                sample_posterior=sample_posterior_eval,
            )
            decoded_text = decode_from_decoder_memory(
                decoder_memory=out.decoder_memory,
                attention_mask=enc['attention_mask'],
            )
            pred_ids_tensor = torch.where(
                torch.sigmoid(out.classification_logits[0]) >= 0.5
            )[0]
            pred_ids = [int(x) for x in pred_ids_tensor.detach().cpu().tolist()]
            pred_names = [emotion_names[idx] for idx in pred_ids]

            print(f'    sample {i + 1}')
            print(f'      text: {text}')
            print(f'      decoded: {decoded_text}')
            print(f'      gold: {gold_names}')
            print(f'      pred: {pred_names}')
    model.train()
    model.t5_encoder.eval()
    model.t5_decoder.eval()



def compute_per_label_metrics(
    labels_np: np.ndarray,
    probs_np: np.ndarray,
    preds_np: np.ndarray,
    thresholds_np: np.ndarray,
    label_names: Sequence[str],
) -> list[Dict[str, Any]]:
    rows: list[Dict[str, Any]] = []
    n_samples = labels_np.shape[0]

    for idx, name in enumerate(label_names):
        y_true = labels_np[:, idx].astype(np.int64)
        y_prob = probs_np[:, idx].astype(np.float64)
        y_pred = preds_np[:, idx].astype(np.int64)

        tp = int(np.sum((y_true == 1) & (y_pred == 1)))
        fp = int(np.sum((y_true == 0) & (y_pred == 1)))
        fn = int(np.sum((y_true == 1) & (y_pred == 0)))
        tn = int(np.sum((y_true == 0) & (y_pred == 0)))
        support = int(np.sum(y_true))
        negatives = int(n_samples - support)

        precision = safe_div(tp, tp + fp)
        recall = safe_div(tp, tp + fn)
        f1 = safe_div(2.0 * precision * recall, precision + recall)
        specificity = safe_div(tn, tn + fp)
        npv = safe_div(tn, tn + fn)
        fpr = safe_div(fp, fp + tn)
        fnr = safe_div(fn, fn + tp)
        jaccard = safe_div(tp, tp + fp + fn)
        accuracy = safe_div(tp + tn, tp + tn + fp + fn)
        balanced_accuracy = 0.5 * (recall + specificity)
        prevalence = safe_div(support, n_samples)
        predicted_positive_rate = safe_div(int(np.sum(y_pred)), n_samples)
        brier = float(np.mean((y_prob - y_true.astype(np.float64)) ** 2))
        roc_auc = safe_metric_call(roc_auc_score, y_true, y_prob)
        average_precision = safe_metric_call(average_precision_score, y_true, y_prob)

        mcc_num = (tp * tn) - (fp * fn)
        mcc_den = math.sqrt(max((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn), 0.0))
        mcc = safe_div(mcc_num, mcc_den)

        rows.append(
            {
                'label_index': idx,
                'label_name': name,
                'threshold': float(thresholds_np[idx]),
                'support': support,
                'negatives': negatives,
                'prevalence': prevalence,
                'predicted_positive_count': int(np.sum(y_pred)),
                'predicted_positive_rate': predicted_positive_rate,
                'tp': tp,
                'fp': fp,
                'fn': fn,
                'tn': tn,
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'specificity': specificity,
                'npv': npv,
                'fpr': fpr,
                'fnr': fnr,
                'jaccard': jaccard,
                'accuracy': accuracy,
                'balanced_accuracy': balanced_accuracy,
                'mcc': mcc,
                'roc_auc': roc_auc,
                'average_precision': average_precision,
                'brier_score': brier,
                'mean_probability': float(np.mean(y_prob)),
                'positive_mean_probability': float(np.mean(y_prob[y_true == 1])) if np.any(y_true == 1) else None,
                'negative_mean_probability': float(np.mean(y_prob[y_true == 0])) if np.any(y_true == 0) else None,
            }
        )
    return rows



def compute_aggregate_metrics(
    labels_np: np.ndarray,
    probs_np: np.ndarray,
    preds_np: np.ndarray,
    per_label_metrics: list[Dict[str, Any]],
) -> Dict[str, Any]:
    aggregate: Dict[str, Any] = {
        'num_samples': int(labels_np.shape[0]),
        'num_labels': int(labels_np.shape[1]),
        'label_cardinality_true': float(labels_np.sum(axis=1).mean()),
        'label_cardinality_pred': float(preds_np.sum(axis=1).mean()),
        'label_density_true': float(labels_np.sum(axis=1).mean() / max(labels_np.shape[1], 1)),
        'label_density_pred': float(preds_np.sum(axis=1).mean() / max(preds_np.shape[1], 1)),
        'samples_with_no_true_labels': int((labels_np.sum(axis=1) == 0).sum()),
        'samples_with_no_predicted_labels': int((preds_np.sum(axis=1) == 0).sum()),
        'subset_accuracy': safe_metric_call(accuracy_score, labels_np, preds_np),
        'zero_one_loss': safe_metric_call(zero_one_loss, labels_np, preds_np),
        'hamming_loss': safe_metric_call(hamming_loss, labels_np, preds_np),
        'jaccard_micro': safe_metric_call(jaccard_score, labels_np, preds_np, average='micro', zero_division=0),
        'jaccard_macro': safe_metric_call(jaccard_score, labels_np, preds_np, average='macro', zero_division=0),
        'jaccard_weighted': safe_metric_call(jaccard_score, labels_np, preds_np, average='weighted', zero_division=0),
        'jaccard_samples': safe_metric_call(jaccard_score, labels_np, preds_np, average='samples', zero_division=0),
        'roc_auc_micro': safe_metric_call(roc_auc_score, labels_np, probs_np, average='micro'),
        'roc_auc_macro': safe_metric_call(roc_auc_score, labels_np, probs_np, average='macro'),
        'roc_auc_weighted': safe_metric_call(roc_auc_score, labels_np, probs_np, average='weighted'),
        'roc_auc_samples': safe_metric_call(roc_auc_score, labels_np, probs_np, average='samples'),
        'average_precision_micro': safe_metric_call(average_precision_score, labels_np, probs_np, average='micro'),
        'average_precision_macro': safe_metric_call(average_precision_score, labels_np, probs_np, average='macro'),
        'average_precision_weighted': safe_metric_call(average_precision_score, labels_np, probs_np, average='weighted'),
        'average_precision_samples': safe_metric_call(average_precision_score, labels_np, probs_np, average='samples'),
        'label_ranking_average_precision': safe_metric_call(label_ranking_average_precision_score, labels_np, probs_np),
        'label_ranking_loss': safe_metric_call(label_ranking_loss, labels_np, probs_np),
        'coverage_error': safe_metric_call(coverage_error, labels_np, probs_np),
        'macro_brier_score': float(np.mean([row['brier_score'] for row in per_label_metrics])),
        'mean_probability': float(np.mean(probs_np)),
        'mean_positive_probability': float(np.mean(probs_np[labels_np == 1])) if np.any(labels_np == 1) else None,
        'mean_negative_probability': float(np.mean(probs_np[labels_np == 0])) if np.any(labels_np == 0) else None,
    }

    for average in ('micro', 'macro', 'weighted', 'samples'):
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels_np,
            preds_np,
            average=average,
            zero_division=0,
        )
        aggregate[f'precision_{average}'] = float(precision)
        aggregate[f'recall_{average}'] = float(recall)
        aggregate[f'f1_{average}'] = float(f1)

    if aggregate['hamming_loss'] is not None:
        aggregate['hamming_acc'] = float(1.0 - aggregate['hamming_loss'])
    else:
        aggregate['hamming_acc'] = None

    return aggregate



def build_detailed_evaluation(
    logits: torch.Tensor,
    labels: torch.Tensor,
    thresholds: float | torch.Tensor,
    label_names: Sequence[str],
) -> Dict[str, Any]:
    if isinstance(thresholds, torch.Tensor):
        threshold_tensor = thresholds.detach().cpu().float()
    else:
        threshold_tensor = torch.full((labels.size(1),), float(thresholds), dtype=torch.float32)

    probs = torch.sigmoid(logits.detach().cpu())
    preds = logits_to_preds(logits.detach().cpu(), threshold=threshold_tensor)

    labels_np = labels.detach().cpu().numpy().astype(np.int64)
    probs_np = probs.numpy().astype(np.float64)
    preds_np = preds.numpy().astype(np.int64)
    thresholds_np = threshold_tensor.numpy().astype(np.float64)

    report_text = classification_report(
        labels_np,
        preds_np,
        target_names=list(label_names),
        zero_division=0,
        digits=4,
    )
    report_dict = classification_report(
        labels_np,
        preds_np,
        target_names=list(label_names),
        zero_division=0,
        output_dict=True,
    )

    per_label_metrics = compute_per_label_metrics(
        labels_np=labels_np,
        probs_np=probs_np,
        preds_np=preds_np,
        thresholds_np=thresholds_np,
        label_names=label_names,
    )
    aggregate_metrics = compute_aggregate_metrics(
        labels_np=labels_np,
        probs_np=probs_np,
        preds_np=preds_np,
        per_label_metrics=per_label_metrics,
    )

    tp = int(np.sum((preds_np == 1) & (labels_np == 1)))
    fp = int(np.sum((preds_np == 1) & (labels_np == 0)))
    fn = int(np.sum((preds_np == 0) & (labels_np == 1)))
    tn = int(np.sum((preds_np == 0) & (labels_np == 0)))

    return {
        'logits': logits.detach().cpu(),
        'labels': labels.detach().cpu(),
        'probs': probs,
        'preds': preds,
        'thresholds': threshold_tensor,
        'classification_report_text': report_text,
        'classification_report_dict': report_dict,
        'per_label_metrics': per_label_metrics,
        'aggregate_metrics': aggregate_metrics,
        'global_confusion': {
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'tn': tn,
        },
    }



def evaluate_loader(
    data_loader: DataLoader,
    desc: str,
    threshold: float | torch.Tensor = 0.5,
    tune_thresholds: bool = False,
) -> Dict[str, object]:
    all_logits = []
    all_labels = []

    total_base_loss = 0.0
    total_total_objective = 0.0
    total_recon_loss = 0.0
    total_kl_loss = 0.0
    total_cls_loss = 0.0
    total_tc_loss = 0.0
    steps = 0

    model.eval()
    model.t5_encoder.eval()
    model.t5_decoder.eval()
    with torch.no_grad():
        for batch in tqdm(data_loader, desc=desc, leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                labels=batch['labels'],
                sample_posterior=sample_posterior_eval,
                classification_weight=classification_weight,
                kl_weight=kl_weight,
                recon_weight=recon_weight,
            )

            if vae_training_mode == 'factorvae':
                if discriminator is None:
                    raise RuntimeError('Expected discriminator in factorvae mode.')
                z_for_tc = extract_valid_token_latents(out.vae.z, batch['attention_mask'])
                if z_for_tc.size(0) < 2:
                    tc_loss = out.loss.new_zeros(())
                else:
                    disc_logits_for_vae = discriminator(z_for_tc)
                    tc_loss = (disc_logits_for_vae[:, 0] - disc_logits_for_vae[:, 1]).mean()
            else:
                tc_loss = out.loss.new_zeros(())

            total_objective = out.loss + (tc_weight * tc_loss)

            all_logits.append(out.classification_logits.detach().cpu())
            all_labels.append(batch['labels'].detach().cpu())
            total_base_loss += float(out.loss.detach().cpu().item())
            total_total_objective += float(total_objective.detach().cpu().item())
            total_recon_loss += float(out.loss_terms['reconstruction'].item())
            total_kl_loss += float(out.loss_terms['kl'].item())
            total_cls_loss += float(out.loss_terms['classification'].item())
            total_tc_loss += float(tc_loss.detach().cpu().item())
            steps += 1

    logits = torch.cat(all_logits, dim=0)
    labels = torch.cat(all_labels, dim=0)
    eval_thresholds = tune_per_label_thresholds(logits, labels) if tune_thresholds else threshold
    detailed = build_detailed_evaluation(
        logits=logits,
        labels=labels,
        thresholds=eval_thresholds,
        label_names=emotion_names,
    )
    aggregate = detailed['aggregate_metrics']
    confusion = detailed['global_confusion']

    metrics: Dict[str, object] = {
        'logits': logits,
        'labels': labels,
        'probs': detailed['probs'],
        'preds': detailed['preds'],
        'thresholds': detailed['thresholds'],
        'avg_base_loss': total_base_loss / max(steps, 1),
        'avg_total_objective': total_total_objective / max(steps, 1),
        'avg_recon': total_recon_loss / max(steps, 1),
        'avg_kl': total_kl_loss / max(steps, 1),
        'avg_cls': total_cls_loss / max(steps, 1),
        'avg_tc': total_tc_loss / max(steps, 1),
        'precision': aggregate['precision_micro'],
        'recall': aggregate['recall_micro'],
        'micro_f1': aggregate['f1_micro'],
        'macro_f1': aggregate['f1_macro'],
        'weighted_f1': aggregate['f1_weighted'],
        'samples_f1': aggregate['f1_samples'],
        'exact_match': aggregate['subset_accuracy'],
        'subset_accuracy': aggregate['subset_accuracy'],
        'hamming_acc': aggregate['hamming_acc'],
        'hamming_loss': aggregate['hamming_loss'],
        'jaccard_micro': aggregate['jaccard_micro'],
        'jaccard_macro': aggregate['jaccard_macro'],
        'jaccard_weighted': aggregate['jaccard_weighted'],
        'jaccard_samples': aggregate['jaccard_samples'],
        'roc_auc_micro': aggregate['roc_auc_micro'],
        'roc_auc_macro': aggregate['roc_auc_macro'],
        'roc_auc_weighted': aggregate['roc_auc_weighted'],
        'roc_auc_samples': aggregate['roc_auc_samples'],
        'average_precision_micro': aggregate['average_precision_micro'],
        'average_precision_macro': aggregate['average_precision_macro'],
        'average_precision_weighted': aggregate['average_precision_weighted'],
        'average_precision_samples': aggregate['average_precision_samples'],
        'label_ranking_average_precision': aggregate['label_ranking_average_precision'],
        'label_ranking_loss': aggregate['label_ranking_loss'],
        'coverage_error': aggregate['coverage_error'],
        'macro_brier_score': aggregate['macro_brier_score'],
        'label_cardinality_true': aggregate['label_cardinality_true'],
        'label_cardinality_pred': aggregate['label_cardinality_pred'],
        'label_density_true': aggregate['label_density_true'],
        'label_density_pred': aggregate['label_density_pred'],
        'samples_with_no_true_labels': aggregate['samples_with_no_true_labels'],
        'samples_with_no_predicted_labels': aggregate['samples_with_no_predicted_labels'],
        'classification_report_text': detailed['classification_report_text'],
        'classification_report_dict': detailed['classification_report_dict'],
        'per_label_metrics': detailed['per_label_metrics'],
        'aggregate_metrics': aggregate,
        'global_confusion': confusion,
        'per_label_tp': torch.tensor([row['tp'] for row in detailed['per_label_metrics']], dtype=torch.float32),
        'per_label_fp': torch.tensor([row['fp'] for row in detailed['per_label_metrics']], dtype=torch.float32),
        'per_label_fn': torch.tensor([row['fn'] for row in detailed['per_label_metrics']], dtype=torch.float32),
        'steps': steps,
    }
    return metrics



def tune_per_label_thresholds(
    logits: torch.Tensor,
    labels: torch.Tensor,
    num_points: int = 19,
) -> torch.Tensor:
    candidates = torch.linspace(0.05, 0.95, steps=num_points)
    probs = torch.sigmoid(logits)
    best_thresholds = torch.full((labels.size(1),), 0.5, dtype=torch.float32)

    for label_idx in range(labels.size(1)):
        y_true = labels[:, label_idx]
        best_score = -1.0
        best_t = 0.5
        for t in candidates:
            preds = (probs[:, label_idx] >= t).float()
            tp = float((preds * y_true).sum().item())
            fp = float((preds * (1.0 - y_true)).sum().item())
            fn = float(((1.0 - preds) * y_true).sum().item())
            precision = tp / max(tp + fp, 1e-8)
            recall = tp / max(tp + fn, 1e-8)
            f1 = 2.0 * precision * recall / max(precision + recall, 1e-8)
            if f1 > best_score:
                best_score = f1
                best_t = float(t.item())
        best_thresholds[label_idx] = best_t
    return best_thresholds



def summarize_eval_metrics(prefix: str, metrics: Dict[str, object]) -> None:
    print(
        f"{prefix}steps={metrics['steps']} "
        f"objective={metrics['avg_total_objective']:.4f} "
        f"base_loss={metrics['avg_base_loss']:.4f} "
        f"recon={metrics['avg_recon']:.4f} ({recon_weight * metrics['avg_recon']:.4f}) "
        f"kl={metrics['avg_kl']:.4f} ({kl_weight * metrics['avg_kl']:.4f}) "
        f"cls={metrics['avg_cls']:.4f} "
        f"tc={metrics['avg_tc']:.4f} ({tc_weight * metrics['avg_tc']:.4f}) "
        f"micro_f1={metrics['micro_f1']:.4f} "
        f"macro_f1={metrics['macro_f1']:.4f} "
        f"weighted_f1={metrics['weighted_f1']:.4f} "
        f"samples_f1={metrics['samples_f1']:.4f} "
        f"subset_acc={metrics['subset_accuracy']:.4f} "
        f"hamming_acc={metrics['hamming_acc']:.4f} "
        f"jaccard_micro={metrics['jaccard_micro']:.4f} "
        f"roc_auc_micro={metrics['roc_auc_micro'] if metrics['roc_auc_micro'] is not None else float('nan'):.4f} "
        f"ap_micro={metrics['average_precision_micro'] if metrics['average_precision_micro'] is not None else float('nan'):.4f} "
        f"lrap={metrics['label_ranking_average_precision'] if metrics['label_ranking_average_precision'] is not None else float('nan'):.4f}"
    )



def print_full_classification_report(prefix: str, metrics: Dict[str, object]) -> None:
    print(f"{prefix}classification report")
    print(metrics['classification_report_text'])
    print(f"{prefix}global confusion: {metrics['global_confusion']}")
    print(f"{prefix}thresholds: {metrics['thresholds'].tolist()}")



def make_json_safe_history(history: list[Dict[str, object]]) -> list[Dict[str, object]]:
    return [to_serializable(item) for item in history]



def save_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(to_serializable(payload), indent=2))



def save_history_json(path: Path, history: list[Dict[str, object]]) -> None:
    save_json(path, make_json_safe_history(history))



def save_per_label_csv(path: Path, rows: list[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text('')
        return
    with path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        for row in rows:
            writer.writerow(to_serializable(row))



def save_eval_artifacts(
    output_dir: Path,
    split_name: str,
    epoch: int,
    metrics: Dict[str, object],
) -> None:
    eval_dir = output_dir / f'{split_name}_epoch_{epoch:03d}'
    eval_dir.mkdir(parents=True, exist_ok=True)

    save_json(eval_dir / 'summary.json', {
        'epoch': epoch,
        'split': split_name,
        'avg_total_objective': metrics['avg_total_objective'],
        'avg_base_loss': metrics['avg_base_loss'],
        'avg_recon': metrics['avg_recon'],
        'avg_kl': metrics['avg_kl'],
        'avg_cls': metrics['avg_cls'],
        'avg_tc': metrics['avg_tc'],
        'aggregate_metrics': metrics['aggregate_metrics'],
        'global_confusion': metrics['global_confusion'],
        'thresholds': metrics['thresholds'],
    })
    save_json(eval_dir / 'classification_report.json', metrics['classification_report_dict'])
    (eval_dir / 'classification_report.txt').write_text(str(metrics['classification_report_text']))
    save_json(eval_dir / 'per_label_metrics.json', metrics['per_label_metrics'])
    save_per_label_csv(eval_dir / 'per_label_metrics.csv', metrics['per_label_metrics'])
    torch.save(
        {
            'logits': metrics['logits'],
            'labels': metrics['labels'],
            'probs': metrics['probs'],
            'preds': metrics['preds'],
            'thresholds': metrics['thresholds'],
            'aggregate_metrics': to_serializable(metrics['aggregate_metrics']),
            'global_confusion': to_serializable(metrics['global_confusion']),
        },
        eval_dir / 'raw_outputs.pt',
    )



def build_history_row(epoch: int, split_name: str, metrics: Dict[str, object]) -> Dict[str, object]:
    return {
        'epoch': epoch,
        'split': split_name,
        **metrics
    }



def save_full_checkpoint(
    checkpoint_path: Path,
    epoch: int,
    step: int,
    best_val_micro_f1: float,
    current_thresholds: torch.Tensor,
    history: list[Dict[str, object]],
    val_metrics: Optional[Dict[str, object]] = None,
    test_metrics: Optional[Dict[str, object]] = None,
) -> None:
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    excluded_metric_keys = {'logits', 'labels', 'probs', 'preds', 'classification_report_text'}
    payload = {
        'epoch': epoch,
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'discriminator_state_dict': discriminator.state_dict() if discriminator is not None else None,
        'disc_optimizer_state_dict': disc_optimizer.state_dict() if disc_optimizer is not None else None,
        'best_val_micro_f1': best_val_micro_f1,
        'current_thresholds': current_thresholds.detach().cpu(),
        'history': make_json_safe_history(history),
        'config': {
            'model_name': 'google/flan-t5-base',
            'classifier_mode': classifier_mode,
            'vae_training_mode': vae_training_mode,
            'num_scalar_factors': num_scalar_factors,
            'vector_latent_dim': vector_latent_dim,
            'classification_weight': classification_weight,
            'recon_weight': recon_weight,
            'kl_weight': kl_weight,
            'tc_weight': tc_weight,
            'sample_posterior_train': sample_posterior_train,
            'sample_posterior_eval': sample_posterior_eval,
            'num_labels': num_labels,
            'emotion_names': emotion_names,
            'max_length': 48,
        },
        'val_metrics': {
            k: (v.detach().cpu() if isinstance(v, torch.Tensor) else to_serializable(v))
            for k, v in (val_metrics or {}).items()
            if k not in excluded_metric_keys
        },
        'test_metrics': {
            k: (v.detach().cpu() if isinstance(v, torch.Tensor) else to_serializable(v))
            for k, v in (test_metrics or {}).items()
            if k not in excluded_metric_keys
        },
    }
    torch.save(payload, checkpoint_path)

In [12]:
num_epochs = 80
threshold = 0.5
evaluation_every = 5
show_eval_samples_every = 5
step = 0
best_thresholds = torch.full((num_labels,), 0.5, dtype=torch.float32)
best_val_micro_f1 = float("-inf")
latest_val_metrics: Optional[Dict[str, object]] = None
history: list[Dict[str, object]] = []
checkpoint_root = Path("factorvae_v8_artifacts")
checkpoint_root.mkdir(parents=True, exist_ok=True)
tokenizer_dir = checkpoint_root / "tokenizer"
tokenizer_dir.mkdir(parents=True, exist_ok=True)
tokenizer.save_pretrained(tokenizer_dir)
print(f"checkpoint_root={checkpoint_root.resolve()}")

for epoch in range(1, num_epochs + 1):
    model.train()
    model.t5_encoder.eval()
    model.t5_decoder.eval()
    epoch_loss = 0.0
    epoch_recon_loss = 0.0
    epoch_kl_loss = 0.0
    epoch_cls_loss = 0.0
    epoch_tc_loss = 0.0
    epoch_disc_loss = 0.0

    epoch_tp = 0.0
    epoch_fp = 0.0
    epoch_fn = 0.0
    epoch_exact = 0.0
    epoch_hamming = 0.0
    epoch_label_tp = torch.zeros(num_labels, dtype=torch.float32)
    epoch_label_fp = torch.zeros(num_labels, dtype=torch.float32)
    epoch_label_fn = torch.zeros(num_labels, dtype=torch.float32)
    batches = 0

    train_pbar = tqdm(train_loader, desc=f"epoch {epoch}/{num_epochs} [train]", leave=False)
    for batch in train_pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        if batch["input_ids"].size(0) < 2:
            continue

        out = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
            sample_posterior=sample_posterior_train,
            classification_weight=classification_weight,
            kl_weight=kl_weight,
            recon_weight=recon_weight,
        )

        if vae_training_mode == "factorvae":
            if discriminator is None:
                raise RuntimeError("Expected discriminator in factorvae mode.")
            set_requires_grad(discriminator, False)
            z_for_tc = extract_valid_token_latents(out.vae.z, batch["attention_mask"])
            if z_for_tc.size(0) < 2:
                tc_loss = out.loss.new_zeros(())
            else:
                disc_logits_for_vae = discriminator(z_for_tc)
                tc_loss = (disc_logits_for_vae[:, 0] - disc_logits_for_vae[:, 1]).mean()
            total_vae_loss = out.loss + (tc_weight * tc_loss)
        else:
            tc_loss = out.loss.new_zeros(())
            total_vae_loss = out.loss

        optimizer.zero_grad(set_to_none=True)
        total_vae_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if vae_training_mode == "factorvae":
            if discriminator is None or disc_optimizer is None or disc_ce is None:
                raise RuntimeError("Discriminator components are not initialized.")
            set_requires_grad(discriminator, True)

            with torch.no_grad():
                disc_forward = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    sample_posterior=True,
                )
                z_for_disc = extract_valid_token_latents(
                    disc_forward.vae.z,
                    batch["attention_mask"],
                )

            if z_for_disc.size(0) < 2:
                disc_loss = out.loss.new_zeros(())
            else:
                z_detached = z_for_disc.detach()
                z_perm = permute_latent_dims(z_detached)
                disc_real = discriminator(z_detached)
                disc_perm = discriminator(z_perm)

                real_target = torch.zeros(z_detached.size(0), dtype=torch.long, device=device)
                perm_target = torch.ones(z_detached.size(0), dtype=torch.long, device=device)
                disc_loss = 0.5 * (
                    disc_ce(disc_real, real_target) + disc_ce(disc_perm, perm_target)
                )

                disc_optimizer.zero_grad(set_to_none=True)
                disc_loss.backward()
                disc_optimizer.step()
        else:
            disc_loss = out.loss.new_zeros(())

        stats = batch_multilabel_stats(
            logits=out.classification_logits.detach(),
            labels=batch["labels"],
            threshold=threshold,
        )

        epoch_loss += float(total_vae_loss.detach().cpu().item())
        epoch_recon_loss += float(out.loss_terms["reconstruction"].item())
        epoch_kl_loss += float(out.loss_terms["kl"].item())
        epoch_cls_loss += float(out.loss_terms["classification"].item())
        epoch_tc_loss += float(tc_loss.detach().cpu().item())
        epoch_disc_loss += float(disc_loss.detach().cpu().item())

        epoch_tp += stats["tp"]
        epoch_fp += stats["fp"]
        epoch_fn += stats["fn"]
        epoch_exact += stats["exact_match"]
        epoch_hamming += stats["hamming_acc"]
        epoch_label_tp += stats["per_label_tp"]
        epoch_label_fp += stats["per_label_fp"]
        epoch_label_fn += stats["per_label_fn"]

        batches += 1
        step += 1

        precision = epoch_tp / max(epoch_tp + epoch_fp, 1e-8)
        recall = epoch_tp / max(epoch_tp + epoch_fn, 1e-8)
        f1 = 2.0 * precision * recall / max(precision + recall, 1e-8)
        macro_f1 = compute_macro_f1_from_counts(
            per_label_tp=epoch_label_tp,
            per_label_fp=epoch_label_fp,
            per_label_fn=epoch_label_fn,
        )
        train_pbar.set_postfix(
            loss=f"{epoch_loss / batches:.4f}",
            cls=f"{epoch_cls_loss / batches:.4f}",
            recon=f"{epoch_recon_loss / batches:.4f}",
            kl=f"{epoch_kl_loss / batches:.4f}",
            tc=f"{epoch_tc_loss / batches:.4f}",
            micro_f1=f"{f1:.4f}",
            macro_f1=f"{macro_f1:.4f}",
        )

    mean_epoch_loss = epoch_loss / max(batches, 1)
    precision = epoch_tp / max(epoch_tp + epoch_fp, 1e-8)
    recall = epoch_tp / max(epoch_tp + epoch_fn, 1e-8)
    f1 = 2.0 * precision * recall / max(precision + recall, 1e-8)
    macro_f1 = compute_macro_f1_from_counts(
        per_label_tp=epoch_label_tp,
        per_label_fp=epoch_label_fp,
        per_label_fn=epoch_label_fn,
    )
    mean_recon = epoch_recon_loss / max(batches, 1)
    mean_kl = epoch_kl_loss / max(batches, 1)
    mean_tc = epoch_tc_loss / max(batches, 1)
    mean_disc = epoch_disc_loss / max(batches, 1)

    train_summary = {
        "epoch": epoch,
        "split": "train",
        "objective": mean_epoch_loss,
        "recon": mean_recon,
        "kl": mean_kl,
        "cls": epoch_cls_loss / max(batches, 1),
        "tc": mean_tc,
        "disc": mean_disc,
        "precision": precision,
        "recall": recall,
        "micro_f1": f1,
        "macro_f1": macro_f1,
        "exact_match": epoch_exact / max(batches, 1),
        "hamming_acc": epoch_hamming / max(batches, 1),
    }
    history.append(train_summary)

    print(
        f"[epoch {epoch}] mode={vae_training_mode} "
        f"objective={mean_epoch_loss:.4f} "
        f"recon={mean_recon:.4f} ({recon_weight * mean_recon:.4f}) "
        f"kl={mean_kl:.4f} ({kl_weight * mean_kl:.4f}) "
        f"cls={epoch_cls_loss / max(batches, 1):.4f} "
        f"tc={mean_tc:.4f} ({tc_weight * mean_tc:.4f}) "
        f"disc={mean_disc:.4f} "
        f"micro_f1={f1:.4f} macro_f1={macro_f1:.4f} "
        f"exact={epoch_exact / max(batches, 1):.4f} "
        f"hamming_acc={epoch_hamming / max(batches, 1):.4f}"
    )

    if epoch % show_eval_samples_every == 0:
        show_eval_samples_during_training(num_samples=2)

    if epoch % evaluation_every == 0 or epoch == num_epochs:
        latest_val_metrics = evaluate_loader(
            val_loader,
            desc=f"epoch {epoch}/{num_epochs} [validation]",
            threshold=0.5,
            tune_thresholds=True,
        )
        best_thresholds = latest_val_metrics["thresholds"]
        summarize_eval_metrics(prefix=f"[epoch {epoch}] validation ", metrics=latest_val_metrics)
        history.append(
            {
                "epoch": epoch,
                "split": "validation",
                "avg_total_objective": latest_val_metrics["avg_total_objective"],
                "avg_base_loss": latest_val_metrics["avg_base_loss"],
                "avg_recon": latest_val_metrics["avg_recon"],
                "avg_kl": latest_val_metrics["avg_kl"],
                "avg_cls": latest_val_metrics["avg_cls"],
                "avg_tc": latest_val_metrics["avg_tc"],
                "precision": latest_val_metrics["precision"],
                "recall": latest_val_metrics["recall"],
                "micro_f1": latest_val_metrics["micro_f1"],
                "macro_f1": latest_val_metrics["macro_f1"],
                "exact_match": latest_val_metrics["exact_match"],
                "hamming_acc": latest_val_metrics["hamming_acc"],
                "thresholds": latest_val_metrics["thresholds"],
            }
        )

        checkpoint_path = checkpoint_root / f"checkpoint_epoch_{epoch:03d}.pt"
        save_full_checkpoint(
            checkpoint_path=checkpoint_path,
            epoch=epoch,
            step=step,
            best_val_micro_f1=best_val_micro_f1,
            current_thresholds=best_thresholds,
            history=history,
            val_metrics=latest_val_metrics,
        )
        save_history_json(checkpoint_root / "history.json", history)

        current_val_micro_f1 = float(latest_val_metrics["micro_f1"])
        if current_val_micro_f1 > best_val_micro_f1:
            best_val_micro_f1 = current_val_micro_f1
            save_full_checkpoint(
                checkpoint_path=checkpoint_root / "best_checkpoint.pt",
                epoch=epoch,
                step=step,
                best_val_micro_f1=best_val_micro_f1,
                current_thresholds=best_thresholds,
                history=history,
                val_metrics=latest_val_metrics,
            )
            print(f"saved new best checkpoint at epoch {epoch} with val_micro_f1={best_val_micro_f1:.4f}")

if latest_val_metrics is None:
    latest_val_metrics = evaluate_loader(
        val_loader,
        desc="final validation",
        threshold=0.5,
        tune_thresholds=True,
    )
    best_thresholds = latest_val_metrics["thresholds"]
    summarize_eval_metrics(prefix="[final validation] ", metrics=latest_val_metrics)
    print_full_classification_report(prefix="[final validation] ", metrics=latest_val_metrics)
    save_eval_artifacts(
        output_dir=checkpoint_root / "evaluations",
        split_name="validation",
        epoch=num_epochs,
        metrics=latest_val_metrics,
    )

print("best_thresholds=", best_thresholds.tolist())

final_test_metrics = evaluate_loader(
    test_loader,
    desc="test",
    threshold=best_thresholds,
    tune_thresholds=False,
)
summarize_eval_metrics(prefix="[final] test ", metrics=final_test_metrics)
print_full_classification_report(prefix="[final] test ", metrics=final_test_metrics)
save_eval_artifacts(
    output_dir=checkpoint_root / "evaluations",
    split_name="test",
    epoch=num_epochs,
    metrics=final_test_metrics,
)

history.append(build_history_row(epoch=num_epochs, split_name="test", metrics=final_test_metrics))

save_full_checkpoint(
    checkpoint_path=checkpoint_root / "final_checkpoint.pt",
    epoch=num_epochs,
    step=step,
    best_val_micro_f1=best_val_micro_f1,
    current_thresholds=best_thresholds,
    history=history,
    val_metrics=latest_val_metrics,
    test_metrics=final_test_metrics,
)
save_history_json(checkpoint_root / "history.json", history)
print(f"saved final checkpoint to {(checkpoint_root / 'final_checkpoint.pt').resolve()}")

num_examples_to_show = 5
print("\nInput/Output comparisons:")
with torch.no_grad():
    model.eval()
    model.t5_encoder.eval()
    model.t5_decoder.eval()
    for i in range(num_examples_to_show):
        row = test_split[i]
        text = str(row["text"])
        gold_ids = parse_label_ids(row["labels"])
        gold_names = [emotion_names[idx] for idx in gold_ids]

        label_target = torch.zeros((1, num_labels), dtype=torch.float32, device=device)
        for idx in gold_ids:
            if 0 <= idx < num_labels:
                label_target[0, idx] = 1.0

        enc = tokenizer(
            text,
            max_length=48,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        out = model(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            labels=label_target,
            sample_posterior=sample_posterior_eval,
            classification_weight=classification_weight,
            kl_weight=kl_weight,
            recon_weight=recon_weight,
        )

        generated_ids = model.t5_decoder.model.generate(
            encoder_outputs=BaseModelOutput(last_hidden_state=out.decoder_memory),
            attention_mask=enc["attention_mask"],
            max_new_tokens=20,
            num_beams=1,
            do_sample=False,
        )
        decoded_text = tokenizer.batch_decode(
            generated_ids.detach().cpu(),
            skip_special_tokens=True,
        )[0]

        pred_ids_tensor = torch.where(
            torch.sigmoid(out.classification_logits[0]) >= best_thresholds.to(device)
        )[0]
        pred_ids = [int(x) for x in pred_ids_tensor.detach().cpu().tolist()]
        pred_names = [emotion_names[idx] for idx in pred_ids]

        print(f"\nExample {i + 1}")
        print(f"Text: {text}")
        print(f"Decoded result: {decoded_text}")
        print(f"Gold emotions: {gold_names}")
        print(f"Predicted emotions: {pred_names}")
        print(f"Example VAE reconstruction loss: {float(out.loss_terms['reconstruction'].item()):.4f}")
        print(f"Example VAE KL loss: {float(out.loss_terms['kl'].item()):.4f}")
        print(f"Example classification loss: {float(out.loss_terms['classification'].item()):.4f}")

checkpoint_root=/mnt/disk1/Projects/NLP-Latent-Learning/factorvae_v8_artifacts


epoch 1/80 [train]:   0%|          | 0/1357 [00:00<?, ?it/s]

KeyboardInterrupt: 